In [90]:
import pandas as pd

In [91]:
nhtsa = pd.read_csv("Downloaded-NHTSA-Ratings-Database.csv", low_memory=False)
df = pd.read_csv("DataSet4-MPGEnriched-Listings.csv", low_memory=False)

In [92]:
nhtsa = nhtsa[(nhtsa['MODEL_YR'] >= 1995) & (nhtsa['MODEL_YR'] <= 2009)]

In [93]:
nhtsa['MAKE'] = nhtsa['MAKE'].str.title().str.strip()
nhtsa['MODEL'] = nhtsa['MODEL'].str.title().str.strip()

In [94]:
safety = nhtsa.groupby(['MAKE', 'MODEL']).agg(
    avg_driver_stars=('FRNT_DRIV_STARS', 'mean'),
    avg_passenger_stars=('FRNT_PASS_STARS', 'mean')
).reset_index()

In [95]:
safety['average_safety_rating'] = ( safety['avg_driver_stars'] + safety['avg_passenger_stars']) / 2

safety = safety.rename(columns={'MAKE': 'make', 'MODEL': 'model'})
safety['average_safety_rating'] = safety['average_safety_rating'].round(2)

print(safety.head(20))
print(f"Total unique make/model combinations with NHTSA Data: {len(safety)}")

     make            model  avg_driver_stars  avg_passenger_stars  \
0   Acura               Cl               NaN                  NaN   
1   Acura          Integra          4.000000             3.000000   
2   Acura           Legend          3.000000             4.000000   
3   Acura              Mdx          4.875000             4.750000   
4   Acura            Nsx-T               NaN                  NaN   
5   Acura              Rdx          5.000000             5.000000   
6   Acura               Rl          4.454545             4.454545   
7   Acura              Rsx          5.000000             5.000000   
8   Acura              Slx          3.000000             3.000000   
9   Acura               Tl          4.500000             4.500000   
10  Acura              Tsx          5.000000             5.000000   
11   Audi               A3               NaN                  NaN   
12   Audi               A4          4.111111             4.333333   
13   Audi         A4 Avant        

In [ ]:
my_models = set(df[['make', 'model']].apply(lambda x: (x['make'], x['model']), axis=1))
nhtsa_models = set(safety[['make', 'model']].apply(lambda x: (x['make'], x['model']), axis=1))

print("In my listings but NOT in NHTSA:")
for pair in sorted(my_models - nhtsa_models):
    print(pair)

print("\nIn NHTSA but NOT in my listings:")
for pair in sorted(nhtsa_models - my_models):
    print(pair)

In my listings but NOT in NHTSA:
('Bmw', '128I')
('Bmw', '323I')
('Bmw', '325Xi')
('Bmw', '328Xi')
('Bmw', '330Ci')
('Bmw', '335I')
('Bmw', '5-Series')
('Bmw', '645Ci')
('Bmw', '650I')
('Bmw', 'Z3')
('Chevrolet', 'C2500')
('Chevrolet', 'K1500')
('Chevrolet', 'W3500')
('Chrysler', '300C')
('Ford', 'E-250')
('Ford', 'E-350')
('Ford', 'E-350 Super Duty')
('Ford', 'E-450')
('Ford', 'F-450')
('Ford', 'Super Duty')
('Honda', 'Prelude')
('Hyundai', 'Xg350')
('Infiniti', 'Fx35')
('Infiniti', 'G35X')
('Infiniti', 'M35')
('Infiniti', 'M35X')
('Kia', 'Spectra5')
('Lexus', 'Es')
('Lexus', 'Gs')
('Lexus', 'Gx')
('Lexus', 'Is')
('Lexus', 'Ls')
('Lexus', 'Rx')
('Lexus', 'Sc')
('Lincoln', 'Mark Viii')
('Scion', 'Tc')
('Scion', 'Xa')
('Scion', 'Xb')
('Scion', 'Xd')
('Volkswagen', 'Beetle')

In NHTSA but NOT in my listings:
('Acura', 'Cl')
('Acura', 'Integra')
('Acura', 'Legend')
('Acura', 'Nsx-T')
('Acura', 'Rsx')
('Acura', 'Slx')
('Audi', 'A4 Avant')
('Audi', 'A4 Cabriolet')
('Audi', 'A4/S4')
('Audi',

In [ ]:
lexus_map = {
    'Es300': 'Es', 'Es330': 'Es', 'Es350': 'Es',
    'Gs300': 'Gs', 'Gs300/430': 'Gs', 'Gs350/430': 'Gs',
    'Gs350/460': 'Gs', 'Gs450H': 'Gs', 'Gs450H Hybrid': 'Gs',
    'Gx470': 'Gx',
    'Is300': 'Is', 'Is250/350': 'Is', 'Is F': 'Is', 'Is300 Sportcross': 'Is',
    'Ls430': 'Ls', 'Ls460/460L': 'Ls', 'Ls600Hl': 'Ls', 'Ls600Hl Hybrid': 'Ls',
    'Rx300': 'Rx', 'Rx330': 'Rx', 'Rx350': 'Rx', 'Rx400H': 'Rx',
    'Sc430': 'Sc',
}

infiniti_map = {
    'Fx35/45': 'Fx35', 'Fx35/50': 'Fx35',
    'G35 Coupe': 'G35', 'G35 Sedan': 'G35',
    'M35/45': 'M35',
}

vw_map = {'New Beetle': 'Beetle'}

nhtsa.loc[nhtsa['MAKE'] == 'Lexus', 'MODEL'] = nhtsa.loc[nhtsa['MAKE'] == 'Lexus', 'MODEL'].replace(lexus_map)
nhtsa.loc[nhtsa['MAKE'] == 'Infiniti', 'MODEL'] = nhtsa.loc[nhtsa['MAKE'] == 'Infiniti', 'MODEL'].replace(infiniti_map)
nhtsa.loc[nhtsa['MAKE'] == 'Volkswagen', 'MODEL'] = nhtsa.loc[nhtsa['MAKE'] == 'Volkswagen', 'MODEL'].replace(vw_map)

In [ ]:
scion_map = {'Tc': 'Scion Tc', 'Xa': 'Scion Xa', 'Xb': 'Scion Xb', 'Xd': 'Scion Xd'}
for my_model, nhtsa_model in scion_map.items():
    mask = (df['make'] == 'Scion') & (df['model'] == my_model)
    df.loc[mask, 'make'] = 'Toyota'
    df.loc[mask, 'model'] = nhtsa_model

df.loc[(df['make'] == 'Infiniti') & (df['model'] == 'G35X'), 'model'] = 'G35'
df.loc[(df['make'] == 'Infiniti') & (df['model'] == 'M35X'), 'model'] = 'M35'

bmw_series_map = {
    '3 Series': ['323I', '325I', '325Xi', '328I', '328Xi', '330Ci', '335I'],
    '5 Series': ['5-Series'],
    '6 Series': ['645Ci', '650I'],
    '128I/135I': ['128I'],
}
bmw_nhtsa = nhtsa[nhtsa['MAKE'] == 'Bmw'].copy()
bmw_series_names = list(bmw_series_map.keys())
bmw_keep = bmw_nhtsa[~bmw_nhtsa['MODEL'].isin(bmw_series_names)]
bmw_expanded = []
for series_name, individual_models in bmw_series_map.items():
    series_rows = bmw_nhtsa[bmw_nhtsa['MODEL'] == series_name].copy()
    for ind_model in individual_models:
        rows = series_rows.copy()
        rows['MODEL'] = ind_model
        bmw_expanded.append(rows)

nhtsa = pd.concat([nhtsa[nhtsa['MAKE'] != 'Bmw'], bmw_keep] + bmw_expanded, ignore_index=True)

In [ ]:
safety = nhtsa.groupby(['MAKE', 'MODEL']).agg(
    avg_frnt_driv_stars=('FRNT_DRIV_STARS', 'mean'),
    avg_frnt_pass_stars=('FRNT_PASS_STARS', 'mean')
).reset_index()
safety['avg_safety_stars'] = ((safety['avg_frnt_driv_stars'] + safety['avg_frnt_pass_stars']) / 2).round(2)
safety = safety.rename(columns={'MAKE': 'make', 'MODEL': 'model'})

In [100]:
your_models = set(zip(df['make'], df['model']))
nhtsa_models = set(zip(safety['make'], safety['model']))

matched = your_models & nhtsa_models
unmatched = your_models - nhtsa_models

print(f"Matched: {len(matched)} / {len(your_models)}")
print("\nUnmatched (will get NaN):")
for pair in sorted(unmatched):
    print(pair)

Matched: 278 / 293

Unmatched (will get NaN):
('Bmw', 'Z3')
('Chevrolet', 'C2500')
('Chevrolet', 'K1500')
('Chevrolet', 'W3500')
('Chrysler', '300C')
('Ford', 'E-250')
('Ford', 'E-350')
('Ford', 'E-350 Super Duty')
('Ford', 'E-450')
('Ford', 'F-450')
('Ford', 'Super Duty')
('Honda', 'Prelude')
('Hyundai', 'Xg350')
('Kia', 'Spectra5')
('Lincoln', 'Mark Viii')


In [ ]:
merged = df.merge(safety[['make', 'model', 'avg_safety_stars']], on=['make', 'model'], how='left')

merged.to_csv("DataSet5-Safety-Enriched-Listings.csv", index=False)
print(f"Saved {len(merged)} rows.")
print(f"Listings with safety data: {merged['avg_safety_stars'].notna().sum()}")
print(f"Listings without safety data: {merged['avg_safety_stars'].isna().sum()}")

Saved 1864 rows.
Listings with safety data: 1780
Listings without safety data: 84
